# 섹션3-4. AI를 활용한 Scatter plot, Box Plot, Swarm Plot, Violin Plot

> 강의: [32가지 데이터 시각화 전략 - 비전공자를 위한 기초이론 & 실습](https://www.inflearn.com/course/32-data-visualizatio/dashboard?cid=343563) (반병현) — 전체 15강

- [x] 강의 시청 완료
- [ ] 실습/정리 완료

## 배운 내용

AI(Gemini Canvas)로 **Strip & Box Plot Visualizer** 웹앱을 만들어 학생건강검사 데이터를 그려봤다.
기능 자체는 정상 동작. 그런데 여기서 한 가지 한계가 드러난다.

> **"점이 너무 많으면 점이 중복해서 찍히는 게 의미가 없을 정도로 많아지는 경우가 생길 수 있습니다."**

이게 **과밀(overplotting)** 이다. strip plot은 원본 점을 하나도 버리지 않고 다 찍는 게 장점인데,
점 수가 어느 선을 넘으면 그 장점이 그대로 단점이 된다 — 점들이 서로 겹쳐 **덩어리(blob)** 가 되고,
"어디에 몇 개가 몰려 있는지"라는 정보가 오히려 사라진다.

이번 강의 데이터가 딱 그 조건이다: 초등(학교급별 1) **3,569명**을 세로 한 줄에 찍는다.
겹침이 심해지면 밀도가 높은 구간과 낮은 구간이 똑같이 "꽉 찬 색"으로 보인다.

## 목표 / 재현할 것

과밀이 실제로 어떻게 정보를 죽이는지 눈으로 확인하고, 대응 수단을 하나씩 적용해 비교한다.

1. 과밀 상태 재현 (점 3,569개를 한 줄에)
2. 완화: **투명도(alpha)** 와 **점 크기** 조정
3. 완화: **샘플링** (그룹당 표시 점 수 상한)
4. 전환: 점을 다 보여주기를 포기하고 **분포 요약**(box / violin)으로


## 실습

> **데이터:** 강의 제공 파일 `학생건강검사 결과분석 rawdata_서울_2015.csv` (4,495행).
> 배포 권한이 불확실해 `.gitignore`로 커밋에서 제외했다 — clone한 경우 이 셀은 파일이 없어 실패한다.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_utils import setup, savefig

setup()
rng = np.random.default_rng(0)

In [ ]:
LECTURE_CSV = "../references/lecture_data/학생건강검사 결과분석 rawdata_서울_2015.csv"

df = pd.read_csv(LECTURE_CSV, encoding="cp949")  # 한글 Windows 엑셀 기본 인코딩
df = df[["학교급별", "성별", "학년", "키", "몸무게"]].dropna()

GROUPS = [1, 2]
LABELS = {1: "초등(1)", 2: "중등(2)"}
height = [df.loc[df["학교급별"] == g, "키"].values for g in GROUPS]

print(f"전체 {len(df):,}행")
for g, vals in zip(GROUPS, height):
    print(f"  학교급별 {g} ({LABELS[g]}): {len(vals):,}명")

### 1. 과밀 재현 — 왜 문제인가

In [ ]:
def strip(ax, datasets, *, size, alpha, jitter_w=0.08, color="#4C78A8"):
    """그룹별 값을 지터(좌우 흔들기)를 줘서 점으로 찍는다."""
    for i, vals in enumerate(datasets, 1):
        jitter = rng.normal(0, jitter_w, len(vals))
        ax.scatter(i + jitter, vals, s=size, alpha=alpha, color=color, linewidths=0)
    ax.set_xticks(range(1, len(datasets) + 1), [LABELS[g] for g in GROUPS])


fig, ax = plt.subplots(figsize=(6, 5))
strip(ax, height, size=25, alpha=1.0)
ax.set(ylabel="키(cm)", title=f"기본 strip plot — 총 {len(df):,}점")
plt.show()

> 초등 쪽이 **속이 꽉 찬 기둥**이 됐다. 점 3,569개가 겹쳐서, 가장 흔한 키 구간과
> 드문 구간이 구분되지 않는다. 점을 다 찍었는데 정작 **분포를 못 읽는** 상태.

### 2. 완화 — 투명도와 점 크기

In [ ]:
settings = [
    (25, 1.00, "기본 (s=25, alpha=1.0)"),
    (12, 0.30, "점 작게 + 반투명"),
    (6, 0.10, "더 작게 + 더 투명"),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, (size, alpha, title) in zip(axes, settings):
    strip(ax, height, size=size, alpha=alpha)
    ax.set_title(title, fontsize=10)
axes[0].set_ylabel("키(cm)")
fig.suptitle("투명도를 낮추면 겹친 곳일수록 진해진다 = 밀도가 보이기 시작")
plt.show()

> **핵심:** alpha를 낮추면 점이 겹칠수록 색이 누적돼 진해진다. 즉 **겹침 자체가 밀도 정보로 바뀐다.**
> 다만 너무 낮추면 드문 값(꼬리)이 아예 안 보이게 되므로, 밀집 구간과 꼬리 사이의 트레이드오프다.

### 3. 완화 — 샘플링 (표시 점 수 상한)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)

for ax, n_max in zip(axes, [None, 500, 150]):
    shown = [
        vals if n_max is None else rng.choice(vals, min(n_max, len(vals)), replace=False)
        for vals in height
    ]
    strip(ax, shown, size=14, alpha=0.45)
    total = sum(len(s) for s in shown)
    ax.set_title("전체" if n_max is None else f"그룹당 최대 {n_max}점", fontsize=10)
    ax.text(0.5, 0.02, f"표시 {total:,}점", transform=ax.transAxes, ha="center", fontsize=9, color="gray")

axes[0].set_ylabel("키(cm)")
fig.suptitle("무작위 샘플링 — 모양은 유지하면서 점 수만 줄인다")
plt.show()

> 분포의 **모양**은 샘플링해도 거의 유지된다. 대신 "몇 명인지"는 그림에서 사라지므로
> **n을 라벨이나 캡션에 반드시 같이 적어야 한다.** (위 그림에도 표시 점 수를 적어뒀다)

### 4. 전환 — 점을 포기하고 분포 요약으로

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)

# (1) 박스플롯 — 요약값 5개
axes[0].boxplot(height, tick_labels=[LABELS[g] for g in GROUPS])
axes[0].set_title("Box — 요약값만", fontsize=10)

# (2) 바이올린 — 분포 모양
parts = axes[1].violinplot(height, showmedians=True)
axes[1].set_xticks([1, 2], [LABELS[g] for g in GROUPS])
axes[1].set_title("Violin — 분포 모양", fontsize=10)

# (3) 바이올린 + 샘플링한 점 (실무에서 가장 무난한 절충)
parts = axes[2].violinplot(height, showmedians=True)
for pc in parts["bodies"]:
    pc.set_alpha(0.25)
sampled = [rng.choice(v, min(200, len(v)), replace=False) for v in height]
strip(axes[2], sampled, size=8, alpha=0.35, jitter_w=0.05, color="#E45756")
axes[2].set_title("Violin + 샘플링 점 (절충)", fontsize=10)

axes[0].set_ylabel("키(cm)")
fig.suptitle("점 수가 많아지면 '원본 다 보여주기'를 포기하는 게 정답일 때가 있다")
plt.show()

### 5. 강의 설명 보충 — "표본이 많으면 정규분포"는 정확하지 않다

강의에서 이렇게 설명한다.

> 점의 개수가 많다는 말은 표본 개수가 많다는 뜻이고, 표본 개수가 많을 때에는 **중심극한정리**에 의해서
> 대부분의 분포가 **정규분포를 따라가게** 되어 있습니다. 그래서 박스 플롯이나 바이올린 플롯 같은
> 통계적 경향성만 보여주는 그래프를 써도 사실 정확합니다.

**실무 결론(점이 많으면 box/violin, 적으면 strip)은 맞다.** 다만 그 근거로 든 중심극한정리는
이 상황에 적용되지 않는다. 헷갈리기 쉬운 부분이라 짚고 넘어간다.

#### 중심극한정리가 실제로 말하는 것

| | 대상 | 내용 |
| --- | --- | --- |
| ⭕ CLT가 말하는 것 | **표본평균**의 분포 | 표본을 여러 번 뽑아 **각 표본의 평균**을 모으면, 그 평균들의 분포가 정규분포에 가까워진다 |
| ❌ CLT가 말하지 않는 것 | **원자료**의 분포 | 데이터를 많이 모은다고 원자료 자체가 정규분포가 되지는 **않는다** |

데이터를 더 모으면 정규분포에 가까워지는 게 아니라, **원래 모집단의 진짜 모양이 더 또렷해진다.**
모집단이 치우쳐 있었다면 표본이 늘수록 치우침이 **더 분명해진다.** 아래에서 이 강의 데이터로 직접 확인한다.

#### 확인 1 — 몸무게는 표본이 늘어도 계속 오른쪽으로 치우쳐 있다

In [ ]:
from scipy import stats

sub = df[df["학교급별"] == 1]["몸무게"].values  # 초등 3,569명

print(f"{'표본 수':>8} | {'왜도(skew)':>10}")
print("-" * 23)
for n in [50, 200, 1000, len(sub)]:
    s = rng.choice(sub, n, replace=False)
    print(f"{n:>8,} | {stats.skew(s):>10.3f}")

print("\n왜도 0 = 좌우대칭(정규분포). 표본이 늘어도 1.0 근처에서 줄지 않는다.")

> 표본을 70배 늘려도 왜도는 사라지지 않는다. 오히려 **모집단이 원래 오른쪽으로 치우쳐 있다는 사실이
> 더 또렷해질 뿐**이다. 몸무게는 아래로는 한계가 있고 위로는 꼬리가 긴 전형적인 비대칭 분포다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, (col, g, label) in zip(axes, [("몸무게", 1, "초등 몸무게"), ("키", 1, "초등 키")]):
    v = df[df["학교급별"] == g][col].values
    ax.hist(v, bins=40, density=True, color="#4C78A8", alpha=0.55, edgecolor="white", label="실제 분포")

    # 같은 평균·표준편차를 갖는 정규분포를 겹쳐 그린다
    xs = np.linspace(v.min(), v.max(), 200)
    ax.plot(xs, stats.norm.pdf(xs, v.mean(), v.std()), color="#E45756", lw=2.5,
            label="같은 평균·표준편차의 정규분포")

    ax.set(xlabel=col, ylabel="밀도",
           title=f"{label} (n={len(v):,}) — 왜도 {stats.skew(v):.2f}, 첨도 {stats.kurtosis(v):.2f}")
    ax.legend(fontsize=9)

fig.suptitle("표본 3,569개 — 그런데도 정규분포와 모양이 다르다")
plt.tight_layout()
plt.show()

> **몸무게**(왼쪽): 오른쪽 꼬리가 길어 정규분포 곡선을 벗어난다.
> **키**(오른쪽): 정규분포보다 **납작하다**(첨도 음수). 초등 1~6학년을 한데 섞었기 때문이다 —
> 7세와 12세의 키는 애초에 다른 분포이고, 그걸 합치면 정규분포가 아니라 **섞인 분포**가 된다.

#### 확인 2 — '섞인 분포'는 표본이 늘수록 오히려 더 잘 드러난다

In [ ]:
elem = df[df["학교급별"] == 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# 합쳐서 보면: 그냥 넓고 납작한 분포
ax1.hist(elem["키"], bins=45, color="#4C78A8", alpha=0.65, edgecolor="white")
ax1.set(xlabel="키(cm)", ylabel="빈도", title=f"초등 전체 (n={len(elem):,}) — 한 덩어리로 보임")

# 학년별로 쪼개면: 6개의 서로 다른 분포였다
for grade in sorted(elem["학년"].unique()):
    v = elem.loc[elem["학년"] == grade, "키"]
    ax2.hist(v, bins=22, alpha=0.5, label=f"{grade}학년 (n={len(v)})")
ax2.set(xlabel="키(cm)", ylabel="빈도", title="학년별로 나누면 — 6개 분포의 혼합")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

> 왼쪽 그림만 보고 "표본이 많으니 정규분포겠지"라고 넘어가면, 사실은 **6개 집단이 섞여 있다**는
> 구조를 놓친다. 표본이 많다는 건 정규분포라는 뜻이 아니라 **구조를 볼 수 있을 만큼 데이터가 있다**는 뜻이다.

#### 확인 3 — 진짜 중심극한정리는 이렇게 작동한다

In [ ]:
sub = df[df["학교급별"] == 1]["몸무게"].values  # 왜도 1.0의 치우친 원자료

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (1) 원자료 — 치우쳐 있다
axes[0].hist(sub, bins=40, color="#4C78A8", alpha=0.7, edgecolor="white")
axes[0].set(xlabel="몸무게(kg)", ylabel="빈도",
            title=f"① 원자료 n={len(sub):,}\n왜도 {stats.skew(sub):.2f} — 정규분포 아님")

# (2)(3) 표본평균의 분포 — n을 키울수록 정규분포에 가까워진다
for ax, n in zip(axes[1:], [5, 30]):
    means = [rng.choice(sub, n, replace=False).mean() for _ in range(3000)]
    ax.hist(means, bins=40, density=True, color="#54A24B", alpha=0.7, edgecolor="white")
    xs = np.linspace(min(means), max(means), 200)
    ax.plot(xs, stats.norm.pdf(xs, np.mean(means), np.std(means)), color="#E45756", lw=2.5)
    ax.set(xlabel=f"{n}명씩 뽑은 표본의 평균(kg)",
           title=f"② 표본평균 분포 (n={n}, 3000회)\n왜도 {stats.skew(means):.2f} — 정규분포에 근접")

fig.suptitle("CLT가 정규분포로 만드는 것은 '원자료'가 아니라 '표본평균'이다")
plt.tight_layout()
plt.show()

> ①의 원자료는 왜도 1.0으로 치우쳐 있는데, 거기서 뽑은 **표본평균들**(②③)은 정규분포에 가까워진다.
> 이게 중심극한정리다. 원자료를 아무리 많이 모아도 ①은 ②가 되지 않는다.

#### 그래서 실무 결론이 어떻게 달라지나

근거가 틀리면 선택도 틀어진다. 여기서는 **box와 violin 중 무엇을 고르느냐**가 갈린다.

| 근거 | 이어지는 선택 | 위험 |
| --- | --- | --- |
| ❌ "표본 많으니 정규분포다" | **박스플롯**이면 충분 (요약값 5개로 정규분포를 다 설명하므로) | 치우침·다봉을 통째로 놓친다 |
| ⭕ "표본이 많아 과밀하다. 분포 모양은 알 수 없다" | **바이올린**을 우선 (모양을 보여주므로) | — |

정규분포를 가정할 수 없기 때문에 **오히려 모양을 보여주는 바이올린이 필요하다.**
박스플롯은 위 초등 몸무게처럼 치우친 분포나 학년이 섞인 분포에서 그 사실을 감춘다.

> **정리:** "점이 많으면 box/violin, 적으면 strip"이라는 강의의 실무 지침은 그대로 유효하다.
> 근거는 중심극한정리가 아니라 **과밀(overplotting)** 이다 — 4번 섹션까지 다룬 내용 그대로.

In [ ]:
# 박스플롯이 무엇을 감추는지 — 같은 데이터, 세 가지 표현
weights = [df.loc[df["학교급별"] == g, "몸무게"].values for g in GROUPS]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)

axes[0].boxplot(weights, tick_labels=[LABELS[g] for g in GROUPS])
axes[0].set_title("Box — 치우침이 안 보인다", fontsize=10)

parts = axes[1].violinplot(weights, showmedians=True)
axes[1].set_xticks([1, 2], [LABELS[g] for g in GROUPS])
axes[1].set_title("Violin — 위로 긴 꼬리가 보인다", fontsize=10)

for i, v in enumerate(weights, 1):
    axes[2].hist(v, bins=35, orientation="horizontal", alpha=0.55,
                 label=f"{LABELS[GROUPS[i-1]]} (왜도 {stats.skew(v):.2f})")
axes[2].set_title("히스토그램 — 모양을 직접 확인", fontsize=10)
axes[2].legend(fontsize=8)

axes[0].set_ylabel("몸무게(kg)")
fig.suptitle("두 그룹 모두 오른쪽으로 치우쳐 있다 — 박스플롯만 보면 알 수 없다")
plt.show()

### 6. 축소 데이터로 바꿨더니 점이 다시 쓸모 있어졌다

강의에서 제공한 **축소본**(`축소된 데이터 건강검진.csv`, 150행)으로 바꾸자 같은 도구·같은 설정인데
점이 읽히기 시작한다. 데이터를 바꾼 것 말고는 아무것도 안 했는데 strip plot이 되살아난 것이다.

**차트 종류가 아니라 점 수가 결정한다** — 앞 섹션들의 결론을 데이터로 직접 확인한 셈이다.

In [ ]:
REDUCED_CSV = "../references/lecture_data/축소된 데이터 건강검진.csv"

red = pd.read_csv(REDUCED_CSV, encoding="cp949")
red = red[["학교급별", "성별", "학년", "키", "몸무게"]].dropna()

print(f"{'':>10} {'전체':>10} {'축소본':>10}")
print("-" * 32)
print(f"{'행 수':>10} {len(df):>10,} {len(red):>10,}")
for g in GROUPS:
    n_full = (df["학교급별"] == g).sum()
    n_red = (red["학교급별"] == g).sum()
    print(f"{LABELS[g]:>10} {n_full:>10,} {n_red:>10,}")

#### 같은 설정, 데이터만 교체

In [ ]:
red_height = [red.loc[red["학교급별"] == g, "키"].values for g in GROUPS]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, (datasets, title) in zip(axes, [(height, "전체 4,495행"), (red_height, "축소본 150행")]):
    strip(ax, datasets, size=14, alpha=0.5)
    n = sum(len(d) for d in datasets)
    ax.set_title(f"{title} — 표시 {n:,}점", fontsize=11)

axes[0].set_ylabel("키(cm)")
fig.suptitle("점 크기·투명도·지터 전부 동일 — 달라진 건 데이터 양뿐")
plt.tight_layout()
plt.show()

> 오른쪽은 점 하나하나가 구분된다. 어디에 몰려 있고 어디가 비었는지, 튀는 값이 몇 개인지가
> 그림에서 바로 읽힌다. 왼쪽에서 사라졌던 정보다.

#### 다만 — 축소본에는 반대쪽 함정이 있다

축소본의 그룹 크기가 고르지 않다. **중등(2)은 22명뿐이다.**

점이 너무 많을 때 violin이 필요했던 것처럼, 점이 **너무 적을 때는 violin이 위험해진다.**
바이올린의 곡선은 KDE(커널 밀도 추정)로 그리는데, 22개 점으로 만든 곡선은
**실제로 있지도 않은 매끄러운 분포를 있는 것처럼 보여준다.**

In [ ]:
g2 = red.loc[red["학교급별"] == 2, "몸무게"].values  # 중등 22명

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)

# (1) 바이올린 단독 — 매끄럽고 자신 있어 보인다
axes[0].violinplot([g2], showmedians=True)
axes[0].set_xticks([1], ["중등(2)"])
axes[0].set_title("Violin 단독 — n을 알 수 없다", fontsize=10)

# (2) 바이올린 + 점 — 실제로는 22개뿐이라는 게 드러난다
parts = axes[1].violinplot([g2], showmedians=True)
for pc in parts["bodies"]:
    pc.set_alpha(0.25)
axes[1].scatter(1 + rng.normal(0, 0.04, len(g2)), g2, s=30, alpha=0.8,
                color="#E45756", zorder=3, linewidths=0)
axes[1].set_xticks([1], ["중등(2)"])
axes[1].set_title(f"Violin + 점 — 실제 {len(g2)}개", fontsize=10)

# (3) 점만 — n이 작을 땐 이게 가장 정직하다
axes[2].scatter(1 + rng.normal(0, 0.04, len(g2)), g2, s=40, alpha=0.85,
                color="#E45756", linewidths=0)
axes[2].hlines(np.median(g2), 0.85, 1.15, color="#4C78A8", lw=2, label=f"중앙값 {np.median(g2):.1f}")
axes[2].set_xticks([1], ["중등(2)"])
axes[2].set_title("Strip 단독 — 가장 정직", fontsize=10)
axes[2].legend(fontsize=8)

axes[0].set_ylabel("몸무게(kg)")
fig.suptitle(f"n={len(g2)}일 때 — 바이올린 곡선은 22개 점이 뒷받침하지 못하는 모양까지 그린다")
plt.show()

> 왼쪽 바이올린만 보면 수백 개 표본으로 그린 것과 구분되지 않는다. **n이 그림에 안 나오기 때문이다.**
> 오른쪽처럼 점을 같이 찍으면 "22개짜리 추정"이라는 사실이 보는 사람에게 그대로 전달된다.

한 가지 더: 축소본에서 그룹별 왜도를 재보면 중등이 **0.10**으로 거의 대칭처럼 나온다
(전체 데이터에서는 0.99였다). 22개로 잰 왜도는 그만큼 흔들린다 — **작은 n에서는 통계량 자체를
믿기 어렵다**는 뜻이고, 이것도 점을 같이 보여줘야 하는 이유다.

In [ ]:
from scipy import stats as _st

print(f"{'몸무게 왜도':>12} {'전체':>10} {'축소본':>10}")
print("-" * 34)
for g in GROUPS:
    a = df.loc[df["학교급별"] == g, "몸무게"].values
    b = red.loc[red["학교급별"] == g, "몸무게"].values
    print(f"{LABELS[g]:>12} {_st.skew(a):>10.2f} {_st.skew(b):>10.2f}   (n={len(a):,} → {len(b)})")

### 강의 결론 — 이 데이터는 바이올린 플롯만 보는 게 가장 깔끔하다

> 학생건강검사 결과분석 데이터 같은 경우는 **바이올린 플롯만 보는 게 제일 깔끔한 것 같아 보입니다.**
> 이렇게 데이터양에 따른 시각화 전략을 보여드렸고 —

**이 선택은 맞다.** 그리고 5번 섹션에서 잰 숫자가 그 이유를 설명해 준다.

| 왜 strip이 아닌가 | 그룹당 3,569점 — 과밀로 덩어리가 된다 (1~4번 섹션) |
| --- | --- |
| **왜 box가 아닌가** | 몸무게 왜도 **1.0**, 초등 키 첨도 **-0.67**. 정규분포가 아니므로 요약값 5개로는 모양이 설명되지 않는다 |
| **왜 violin인가** | 치우침과 봉우리를 그대로 보여준다 — 정규분포를 가정할 수 없을 때 필요한 게 정확히 이것 |

즉 "표본이 많아서 요약 통계로 충분하다"가 아니라, **"모양을 알 수 없으니 모양을 보여주는 그림이
필요하다"** 가 바이올린을 고르는 이유다. 결론은 같지만 근거가 반대 방향이다.

### 정리 — 점 수에 따른 선택

| 점 수(그룹당) | 권장 | 이유 |
| --- | --- | --- |
| ~100 | **strip / swarm** 그대로 | 원본을 다 보여주는 게 가장 정직하고, 겹침도 없다. n이 30 미만이면 violin은 피한다(6번 섹션) |
| 100~1,000 | strip + **alpha 낮추기·점 크기 줄이기** | 겹침이 밀도 정보로 바뀐다 |
| 1,000~ | **violin** (필요하면 샘플 점 겹치기) | 과밀로 점은 못 쓴다. 분포 모양은 여전히 봐야 하므로 box보다 violin |

**두 가지 주의**

- 어떤 방법을 쓰든 **`n`을 표기한다.** 샘플링하면 그림에서 표본 크기 정보가 사라진다.
- 바이올린도 만능은 아니다. 초등 키는 바이올린으로 보면 그냥 "넓은 분포" 한 덩어리인데,
  실제로는 **6개 학년이 섞인 것**이었다(5번 섹션 확인 2). 한 장으로 끝낼 거면 violin이 맞지만,
  구조가 의심되면 **쪼개서 보는 것**(facet)을 한 번은 해봐야 한다.

---

## 도구에 추가한 기능

강의 영상은 점 오버레이 없이 **바이올린 단독**으로 나와서, Gemini에 "강의는 바이올린 플롯 단독으로도
나오는데?"라고 요청해 **`Violin Plot 단독` 모드**를 추가했다.

- KDE 밀도 곡선 + 중앙값/사분위수 선만 그리고 점 오버레이는 끈다
- 버튼을 누르면 `Strip 점 활성화` 체크가 자동으로 해제되고, 필요하면 다시 켤 수 있다
- 도구 부제도 `Strip Plot & Box Plot` → `Strip, Box, Violin & Swarm Plot Analysis`로 바뀜

**단, 6번 섹션의 이유로 이 모드를 기본값으로 쓰는 건 데이터에 따라 위험하다.** 축소본 중등(n=22)처럼
표본이 작을 때는 점을 같이 켜두는 쪽이 정직하다.

## 메모

- Gemini Canvas로 만든 도구는 `references/lecture_data/strip_box_plot_visualizer.html`에 저장해둠
- 과밀은 strip plot만의 문제가 아니다 — 산점도에서도 같은 일이 생기고, 그때 쓰는 게 2D density / hexbin
  (19강 `10_AI_2D밀도_헥스빈.ipynb`에서 다시 나옴)
- 이 강의 데이터는 그룹 크기가 고르지 않다(축소본 128 vs 22). 그룹별로 적정 표현이 다를 수 있다는 점 기억
